### import pkg

In [3]:
import yaml
from jinja2 import Template
from langsmith import Client

from dotenv import load_dotenv
load_dotenv("../../.env")

True

In [ ]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

In [2]:
preprocessed_context = "- a \n- b"
question = "What is a?"

prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b

Question:
What is a?    



### jinja

- better handle json, and more advanced variable handling

In [3]:
jinja_template = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{{ preprocessed_context }}

Question:
{{ question }}    
"""

In [5]:
template = Template(jinja_template)
rendered_template = template.render(preprocessed_context=preprocessed_context, question=question)

In [6]:
print(rendered_template)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b

Question:
What is a?    


In [11]:
# yaml can version it, can add auth, that how production system manage prompt
def prompt_template_config(yaml_path, prompt_key):

    with open(yaml_path, "r") as file:
        config = yaml.safe_load(file)

    print(config['prompts'])
    template_content = config["prompts"][prompt_key]

    template = Template(template_content)

    return template

In [12]:
prompt_template_config('./prompts/retrieval_generation.yml', 'retrieval_generation')

{'retrieval_generation': 'You are a shopping assistant that can answer questions about the products in stock.\n\nYou will be given a question and a list of context.\n\nInstructions:\n- Answer the question based on the provided context only.\n- Never use word context and refer to it as the available products.\n- Do not use markdown formatting.\n\nContext:\n{{ preprocessed_context }}\n\nQuestion:\n{{ question }}  '}


<Template memory:117140ef0>

### prompt registeration


In [21]:
help(ls_client.pull_prompt)

Help on method pull_prompt in module langsmith.client:

pull_prompt(prompt_identifier: 'str', *, include_model: 'bool | None' = False, secrets: 'dict[str, str] | None' = None, secrets_from_env: 'bool' = False, skip_cache: 'bool' = False, dangerously_pull_public_prompt: 'bool' = False) -> 'Any' method of langsmith.client.Client instance
    Pull a prompt and return it as a LangChain `PromptTemplate`.

    This method requires [`langchain-core`](https://pypi.org/project/langchain-core).

    Args:
        prompt_identifier: The identifier of the prompt.
        include_model: Whether to include model configuration in the loaded
            prompt.
        secrets: A map of secrets to use for explicit serialized LangChain secret
            references in the manifest, e.g. `{'OPENAI_API_KEY': 'sk-...'}`.

            If a manifest secret reference is not found in the map, it will be
            loaded from the environment only if `secrets_from_env` is `True`.
            Deserialized mode

In [31]:
from langsmith import Client
client = Client()
for prompt in client.list_prompts():
    print(prompt)

('repos', [Prompt(repo_handle='bug_to_user_story_v2', description='Prompt otimizado para converter relatos de bugs em User Stories com Few-Shot, Role Prompting e Chain of Thought', readme='', id='9a2ea98b-f914-46f7-811e-bc564211d2ff', tenant_id='8d924a4f-bfb7-4a2c-a077-74abbe242330', created_at=datetime.datetime(2026, 7, 3, 20, 21, 9, 65229), updated_at=datetime.datetime(2026, 7, 6, 0, 5, 33, 28434), is_public=True, is_archived=False, tags=['ChatPromptTemplate'], original_repo_id=None, upstream_repo_id=None, owner='pollytheparrotbcesartest', full_name='pollytheparrotbcesartest/bug_to_user_story_v2', num_likes=0, num_downloads=17, num_views=6, liked_by_auth_user=None, last_commit_hash='717c1ad2d111ae9c8f2ec36c09c7182cf7476a5e515867d86adfe2a2ff496e53', num_commits=4, original_repo_full_name=None, upstream_repo_full_name=None), Prompt(repo_handle='bug_to_user_story_v2', description='Prompt otimizado para converter relatos de bugs em User Stories, escalando o nível de detalhe conforme a co

In [35]:
prompt[1]

9800

In [5]:
client = Client()
prompt = client.pull_prompt("retrieval-generation")
prompt

ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'retrieval-generation', 'lc_hub_commit_hash': 'e2cd7724c6448c58a843e76bd938244832f7b3951cf1fb5c183bc6ca99ef12bd'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a shopping assistant that can answer questions about the products in stock.\nYou will be given a question and a list of context.\n\nInstructions:\n- Answer the question based on the provided context only.\n- Never use word context and refer to it as the available products.\n- Do not use markdown formatting.\n\nContext:\n{{ preprocessed_context }}\n\nQuestion:\n{{ question }}    '), additional_kwargs={})])